<a href="https://colab.research.google.com/github/IKKNIGHT/Protein_interactions/blob/main/Phase0_1_Setup_GoldStandard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 0 + Phase 1: Setup and gold-standard builder
Workspace: `My Drive / Protein_interactions` (shortcut). Per PDB this notebook:
1. downloads the complex, keeps protein atoms only (no water / ligand / DNA),
2. proposes which chains are the target and which are the native partner (**you review these**),
3. computes native interface residues two ways (5 Å heavy-atom contact and change in SASA),
4. saves `gold/<id>/target.pdb`, `partner.pdb`, `interface_residues.json`, plus a summary table.

Run top to bottom. Cell 5 stops for review before Cell 6 builds anything.

In [2]:
# ── Cell 1: Mount Drive, point at Protein_interactions, verify read AND write ─
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, time, random, platform, subprocess, re, copy

BASE = '/content/drive/MyDrive/Protein_interactions'

if not os.path.isdir(BASE):
    raise FileNotFoundError(
        f'{BASE} not found. Check the shortcut name is exactly "Protein_interactions" in My Drive. '
        'A freshly added shortcut sometimes needs: drive.mount("/content/drive", force_remount=True)')

print('Shortcut resolves to:', os.path.realpath(BASE))

# A shortcut can be listable but not writable (e.g. you only have viewer access). Test it.
_t = f'{BASE}/.write_test'
try:
    with open(_t, 'w') as f:
        f.write('ok')
    os.remove(_t)
    print('Write test: OK')
except Exception as e:
    raise RuntimeError(
        f'Cannot write to {BASE}: {e}\n'
        'Fix: confirm you have Editor access on the target folder, then force-remount Drive. '
        'If it still fails, create a normal folder in My Drive and use that path instead.')

CONFIG   = f'{BASE}/config'
GOLD     = f'{BASE}/gold'
MANIFEST = f'{BASE}/manifests'
RESULTS  = f'{BASE}/results'
for d in (CONFIG, GOLD, MANIFEST, RESULTS):
    os.makedirs(d, exist_ok=True)

# Raw downloads go to local disk (faster than Drive); only small outputs go to Drive.
LOCAL_RAW = '/content/ppi_local/raw'
os.makedirs(LOCAL_RAW, exist_ok=True)

SEED = 42
random.seed(SEED)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'biopython', 'requests', 'pandas', 'numpy', 'scipy'])
print('\nBase folder:', BASE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Shortcut resolves to: /content/drive/.shortcut-targets-by-id/1s2myX0p8Qedv9Im6oj4LNAzFWbabO4qJ/Protein_interactions
Write test: OK

Base folder: /content/drive/MyDrive/Protein_interactions


In [3]:
# ── Cell 2: Load the tier-A table (writes it to Drive config/ if missing) ─────
import pandas as pd, io

CSV_PATH = f'{CONFIG}/ppi_benchmark_tierA.csv'
CSV_EMBEDDED = """id,target_protein,partner_to_predict,pdb_id,pdb_confidence,partner_size,target_chain,partner_chain,leakage_or_notes
4,CDK2,CDKN1B (p27),1JSU,high,medium disordered,TBD,TBD,p27 wraps CDK2/cyclin A; non-compact binder; cyclin A also in the complex
5,MDM2,TP53 peptide,1YCR,high,peptide ~15 aa,TBD,TBD,classic helix-in-groove; best positive control
6,BRCA1,BARD1,1JM7,high,small RING ~100 aa,TBD,TBD,RING heterodimer helical bundle
7,RAD51,BRCA2 (BRC4),1N0W,high,peptide ~35 aa,TBD,TBD,
8,PCNA,POLD1 PIP peptide,1AXC,proxy (p21),peptide,TBD,TBD,no confident POLD1 ID; 1AXC is p21 PIP proxy
10,BCL2,BAX BH3,2XA0,verify,BH3 helix,TBD,TBD,use BH3 peptide as the partner
11,BCL2L1 (Bcl-xL),BAD BH3,1G5J,likely,BH3 peptide,TBD,TBD,
12,FAS,FADD,3EZQ,high,death domain ~85 aa,TBD,TBD,death domain complex is a large oligomer
13,XIAP (BIR3),CASP9,1NW9,high,peptide/small,TBD,TBD,IAP-binding motif tetrapeptide
20,HRAS,RAF1 RBD,4G0N,likely,small ~80 aa,TBD,TBD,
22,GNAI1,GPSM1 (AGS3),1KJY,verify,GoLoco peptide,TBD,TBD,
24,ARF1,GGA1 (GAT),1J2J,likely,small helical ~70 aa,TBD,TBD,
26,JUN,FOS,1FOS,high,bZIP ~60 aa,TBD,TBD,helical coiled coil; similar fold family to training bHLH-LZ; DNA present in structure
27,MYC,MAX,1NKP,high,~80 aa,TBD,TBD,original positive control; GNN trained without 1NKP
28,CREB1 (KID),CREBBP (KIX),1KDX,high,KIX ~90 aa,TBD,TBD,target KID peptide or target KIX; both fine
30,MEN1,KMT2A (MLL1),4GQ6,likely,peptide,TBD,TBD,
33,VHL,HIF1A,1LM8,high,peptide ~20 aa,TBD,TBD,target is VHL/EloB/EloC
34,KEAP1,NFE2L2 (NRF2),2FLU,likely,peptide ~16 aa,TBD,TBD,
37,CTNNB1,TCF7L2,1JDH,verify,extended peptide,TBD,TBD,
38,VCL,TLN1 (VBS),1T01,likely,helical peptide,TBD,TBD,
45,GRIA2 (C-tail),GRIP1 (PDZ),1N7E,verify,PDZ ~90 aa,TBD,TBD,predict PDZ from tail peptide
"""

if not os.path.exists(CSV_PATH):
    with open(CSV_PATH, 'w') as f:
        f.write(CSV_EMBEDDED)
    print('Wrote embedded tier-A CSV to', CSV_PATH)

pairs = pd.read_csv(CSV_PATH)

# Pairs where the CSV "target" is a short peptide: swap so the structured partner is the design
# target and the peptide is what we predict. Edit this set if you disagree.
SWAP_IDS = {28, 45}
for i in SWAP_IDS:
    m = pairs['id'] == i
    pairs.loc[m, ['target_protein', 'partner_to_predict']] = pairs.loc[m, ['partner_to_predict', 'target_protein']].values

print(pairs[['id', 'target_protein', 'partner_to_predict', 'pdb_id', 'pdb_confidence']].to_string(index=False))
print(f'\nSwapped target/partner for ids {sorted(SWAP_IDS)} (peptide as target is a poor design setup).')

Wrote embedded tier-A CSV to /content/drive/MyDrive/Protein_interactions/config/ppi_benchmark_tierA.csv
 id  target_protein partner_to_predict pdb_id pdb_confidence
  4            CDK2       CDKN1B (p27)   1JSU           high
  5            MDM2       TP53 peptide   1YCR           high
  6           BRCA1              BARD1   1JM7           high
  7           RAD51       BRCA2 (BRC4)   1N0W           high
  8            PCNA  POLD1 PIP peptide   1AXC    proxy (p21)
 10            BCL2            BAX BH3   2XA0         verify
 11 BCL2L1 (Bcl-xL)            BAD BH3   1G5J         likely
 12             FAS               FADD   3EZQ           high
 13     XIAP (BIR3)              CASP9   1NW9           high
 20            HRAS           RAF1 RBD   4G0N         likely
 22           GNAI1       GPSM1 (AGS3)   1KJY         verify
 24            ARF1         GGA1 (GAT)   1J2J         likely
 26             JUN                FOS   1FOS           high
 27             MYC                MAX   1

In [4]:
# ── Cell 3: Name keywords used to propose chains (CSV roles, BEFORE any swap) ──
# id -> (keywords for the CSV "target" entity/entities, keywords for the CSV "partner" entity)
# Multiple target keywords can match multiple entities (e.g. VHL + Elongin B + Elongin C).
# These are my best guesses at RCSB entity names; wrong guesses show up as UNRESOLVED in Cell 5.
ALIASES = {
  4:  (['cell division protein kinase 2', 'cyclin-dependent kinase 2', 'cdk2', 'cyclin-a2', 'cyclin a'],
       ['cyclin-dependent kinase inhibitor 1b', 'p27', 'kip1']),
  5:  (['mdm2', 'e3 ubiquitin-protein ligase mdm2'], ['p53', 'cellular tumor antigen p53']),
  6:  (['breast cancer type 1 susceptibility protein', 'brca1'], ['bard1', 'brca1-associated ring domain protein 1']),
  7:  (['dna repair protein rad51', 'rad51'], ['brca2', 'breast cancer type 2 susceptibility protein']),
  8:  (['proliferating cell nuclear antigen', 'pcna'], ['cyclin-dependent kinase inhibitor 1', 'p21', 'cdkn1a', 'waf1', 'cip1']),
  10: (['apoptosis regulator bcl-2', 'bcl-2', 'bcl2'], ['apoptosis regulator bax', 'bax', 'bcl-2-like protein 4']),
  11: (['bcl-2-like protein 1', 'bcl-x', 'bcl-xl'], ['bcl2-antagonist of cell death', 'bcl-2-binding component', 'bad']),
  12: (['tumor necrosis factor receptor superfamily member 6', 'fas'], ['fas-associated death domain protein', 'fadd']),
  13: (['baculoviral iap repeat-containing protein 4', 'xiap', 'e3 ubiquitin-protein ligase xiap'], ['caspase-9', 'caspase 9', 'casp9']),
  20: (['gtpase hras', 'hras', 'h-ras'], ['raf proto-oncogene serine/threonine-protein kinase', 'raf-1', 'raf1', 'c-raf']),
  22: (['guanine nucleotide-binding protein g(i) subunit alpha-1', 'gnai1', 'g(i) subunit alpha'], ['g-protein-signaling modulator 1', 'ags3', 'gpsm1', 'goloco']),
  24: (['adp-ribosylation factor 1', 'arf1'], ['adp-ribosylation factor-binding protein gga1', 'gga1', 'adaptor protein gga1']),
  26: (['proto-oncogene c-jun', 'c-jun', 'jun'], ['proto-oncogene c-fos', 'c-fos', 'fos']),
  27: (['myc proto-oncogene protein', 'c-myc', 'myc'], ['protein max', 'myc-associated factor x', 'max']),
  28: (['cyclic amp-responsive element-binding protein 1', 'creb1', 'camp response element', 'kid'], ['creb-binding protein', 'cbp', 'crebbp', 'kix']),
  30: (['menin', 'men1'], ['histone-lysine n-methyltransferase 2a', 'kmt2a', 'mll', 'mixed-lineage leukemia']),
  33: (['von hippel-lindau disease tumor suppressor', 'vhl', 'elongin-b', 'elongin-c',
        'transcription elongation factor b polypeptide 1', 'transcription elongation factor b polypeptide 2'],
       ['hypoxia-inducible factor 1-alpha', 'hif-1-alpha', 'hif1a', 'hif-1a']),
  34: (['kelch-like ech-associated protein 1', 'keap1'], ['nuclear factor erythroid 2-related factor 2', 'nrf2', 'nfe2l2']),
  37: (['catenin beta-1', 'beta-catenin'], ['transcription factor 7-like 2', 'tcf7l2', 'tcf-4', 'tcf4', 'htcf-4']),
  38: (['vinculin'], ['talin']),
  45: (['glutamate receptor 2', 'glur2', 'gria2'], ['glutamate receptor-interacting protein 1', 'grip1', 'glutamate receptor interacting protein']),
}
# Note: ids 28 and 45 are swapped by SWAP_IDS in Cell 2; Cell 5 swaps these keyword lists to match.
missing = [i for i in pairs['id'] if i not in ALIASES]
print('Rows with no aliases (will be UNRESOLVED):', missing or 'none')

Rows with no aliases (will be UNRESOLVED): none


In [5]:
# ── Cell 4: Helper functions (RCSB metadata, chain proposal, gold-standard geometry) ──
import requests, numpy as np
from scipy.spatial import cKDTree
from Bio import PDB
from Bio.PDB.SASA import ShrakeRupley
from Bio.Data.IUPACData import protein_letters_3to1

CONTACT_CUTOFF = 5.0     # heavy-atom distance defining a contact (A)
DSASA_MIN      = 1.0     # buried SASA (A^2) to call a residue interface

def fetch_entities(pdb_id):
    # RCSB GraphQL: one record per polymer entity, with description, type, auth chain ids, organism.
    q = ('{ entry(entry_id:"%s"){ polymer_entities{'
         ' rcsb_polymer_entity{pdbx_description}'
         ' entity_poly{rcsb_entity_polymer_type}'
         ' rcsb_entity_source_organism{ncbi_scientific_name}'
         ' rcsb_polymer_entity_container_identifiers{auth_asym_ids} } } }') % pdb_id
    r = requests.post('https://data.rcsb.org/graphql', json={'query': q}, timeout=60)
    r.raise_for_status()
    ents = (r.json().get('data', {}).get('entry') or {}).get('polymer_entities') or []
    out = []
    for e in ents:
        org = e.get('rcsb_entity_source_organism') or []
        out.append({
            'desc':   (e['rcsb_polymer_entity'] or {}).get('pdbx_description', ''),
            'type':   (e['entity_poly'] or {}).get('rcsb_entity_polymer_type', ''),
            'organism': org[0].get('ncbi_scientific_name') if org else None,
            'chains': e['rcsb_polymer_entity_container_identifiers']['auth_asym_ids'],
        })
    return out

def download_structure(pdb_id):
    path = f'{LOCAL_RAW}/{pdb_id}.cif'
    if not os.path.exists(path):
        r = requests.get(f'https://files.rcsb.org/download/{pdb_id}.cif', timeout=120)
        r.raise_for_status()
        open(path, 'wb').write(r.content)
    return path

def load_model0(path):
    return PDB.MMCIFParser(QUIET=True).get_structure('x', path)[0]   # first model only (NMR-safe)

def protein_heavy_atoms(chain):
    atoms = []
    for res in chain:
        if res.id[0] != ' ':          # drops waters / ligands / modified residues (incl. MSE)
            continue
        for a in res:
            if (a.element or '').strip() != 'H' and a.get_altloc() in (' ', 'A'):
                atoms.append(a)
    return atoms

def contacts_between(model, chains_a, chains_b, cutoff=4.5):
    A = [a for c in chains_a if c in model for a in protein_heavy_atoms(model[c])]
    B = [a for c in chains_b if c in model for a in protein_heavy_atoms(model[c])]
    if not A or not B:
        return 0
    ta = cKDTree(np.array([a.coord for a in A]))
    tb = cKDTree(np.array([b.coord for b in B]))
    return int(ta.count_neighbors(tb, cutoff))

def kw_hit(text, kws):
    t = text.lower(); best = 0
    for k in kws:
        if re.search(r'(?<![a-z0-9])' + re.escape(k.lower()) + r'(?![a-z0-9])', t):
            best = max(best, len(k))
    return best

def propose_chains(model, entities, tkw, pkw):
    # Assign each protein entity to target/partner by keyword; if both match, the longer keyword wins.
    t_ents, p_ents = [], []
    for e in entities:
        if 'protein' not in (e['type'] or '').lower():
            continue
        th, ph = kw_hit(e['desc'], tkw), kw_hit(e['desc'], pkw)
        if th == 0 and ph == 0:
            continue
        (t_ents if th >= ph else p_ents).append(e)
    if not t_ents or not p_ents:
        return None, f'no entity matched ({"target" if not t_ents else "partner"})'
    def present(e): return [c for c in e['chains'] if c in model]
    # Choose the (target chain, partner chain) pair with the most contacts: picks one real copy
    # when the asymmetric unit holds several complexes.
    best = (-1, None, None)
    for te in t_ents:
        for tc in present(te):
            for pe in p_ents:
                for pc in present(pe):
                    n = contacts_between(model, [tc], [pc])
                    if n > best[0]:
                        best = (n, tc, pc)
    n0, t0, p0 = best
    if t0 is None or n0 <= 0:
        return None, 'no contacting target/partner chain pair found'
    target_chains, partner_chains, notes = [t0], [p0], []
    for te in t_ents:                       # extra target entities (e.g. cyclin, elongins): best-contacting copy
        if t0 in te['chains']:
            continue
        cand = sorted(((contacts_between(model, [c], [t0, p0]), c) for c in present(te)), reverse=True)
        if cand and cand[0][0] > 0:
            target_chains.append(cand[0][1])
        else:
            notes.append(f'extra target entity "{te["desc"]}" not in contact; skipped')
    for pe in p_ents:
        if p0 in pe['chains']:
            continue
        cand = sorted(((contacts_between(model, [c], [p0] + target_chains), c) for c in present(pe)), reverse=True)
        if cand and cand[0][0] > 0:
            partner_chains.append(cand[0][1])
    return {'target_chains': target_chains, 'partner_chains': partner_chains,
            'contacts_4.5A': n0, 'notes': notes}, 'ok'

class KeepChains(PDB.Select):
    def __init__(self, chains): self.chains = set(chains)
    def accept_chain(self, chain): return chain.id in self.chains
    def accept_residue(self, residue): return residue.id[0] == ' '
    def accept_atom(self, atom): return atom.get_altloc() in (' ', 'A')

def write_chains(model_struct, chains, out_path):
    io = PDB.PDBIO(); io.set_structure(model_struct); io.save(out_path, KeepChains(chains))

def res_key(res):
    return f'{res.get_parent().id}:{res.id[1]}{res.id[2].strip()}'

def seq_of(chain):
    s = []
    for r in chain:
        if r.id[0] == ' ':
            s.append(protein_letters_3to1.get(r.get_resname().capitalize(), 'X'))
    return ''.join(s)

def helix_fraction(chain):
    # Crude: fraction of residues whose CA(i)-CA(i+3) distance is helix-like (4.5-5.6 A). Not DSSP.
    cas = [r['CA'].coord for r in chain if r.id[0] == ' ' and 'CA' in r]
    if len(cas) < 5: return None
    d = [np.linalg.norm(cas[i] - cas[i + 3]) for i in range(len(cas) - 3)]
    return float(np.mean([4.5 <= x <= 5.6 for x in d]))

def numbering_gaps(chain):
    nums = [r.id[1] for r in chain if r.id[0] == ' ']
    return [(a, b) for a, b in zip(nums, nums[1:]) if b - a > 1]

def per_residue_sasa(model):
    s = PDB.Structure.Structure('s'); s.add(model.copy())
    ShrakeRupley(probe_radius=1.4, n_points=100).compute(s, level='R')
    return {res_key(r): r.sasa for r in s[0].get_residues() if r.id[0] == ' '}

def build_gold(pdb_id, t_chains, p_chains, out_dir):
    cif = download_structure(pdb_id)
    struct = PDB.MMCIFParser(QUIET=True).get_structure(pdb_id, cif)
    os.makedirs(out_dir, exist_ok=True)
    write_chains(struct, t_chains,            f'{out_dir}/target.pdb')
    write_chains(struct, p_chains,            f'{out_dir}/partner.pdb')
    write_chains(struct, t_chains + p_chains, f'{out_dir}/complex_clean.pdb')
    P = PDB.PDBParser(QUIET=True)
    tgt, par, cpx = (P.get_structure(n, f'{out_dir}/{n}.pdb')[0] for n in ('target', 'partner', 'complex_clean'))

    # 1) Heavy-atom contact definition (target residues within CONTACT_CUTOFF of any partner atom)
    p_atoms = [a for c in par for a in protein_heavy_atoms(c)]
    ptree = cKDTree(np.array([a.coord for a in p_atoms]))
    contact, n_contact_atoms = [], 0
    for c in tgt:
        for r in c:
            if r.id[0] != ' ': continue
            pts = [a.coord for a in r if (a.element or '').strip() != 'H']
            hit = ptree.query_ball_point(pts, CONTACT_CUTOFF) if pts else []
            if any(len(h) for h in hit):
                contact.append(res_key(r)); n_contact_atoms += sum(1 for h in hit if len(h))
    # partner-side interface (for later pose comparison)
    ttree = cKDTree(np.array([a.coord for c in tgt for a in protein_heavy_atoms(c)]))
    partner_iface = []
    for c in par:
        for r in c:
            if r.id[0] != ' ': continue
            pts = [a.coord for a in r if (a.element or '').strip() != 'H']
            if pts and any(len(h) for h in ttree.query_ball_point(pts, CONTACT_CUTOFF)):
                partner_iface.append(res_key(r))

    # 2) Buried-SASA definition: SASA(target alone) - SASA(target in complex)
    alone = per_residue_sasa(tgt); cpx_s = per_residue_sasa(cpx)
    dsasa = {k: alone[k] - cpx_s[k] for k in alone if k in cpx_s}
    dsasa_set = sorted(k for k, v in dsasa.items() if v > DSASA_MIN)

    consensus = sorted(set(contact) & set(dsasa_set))
    flags = []
    if not contact: flags.append('NO_CONTACT')
    if len(consensus) < 8: flags.append('SMALL_INTERFACE')
    if sum(1 for c in par for r in c if r.id[0] == ' ') < 5: flags.append('TINY_PARTNER')

    info = {
        'pdb_id': pdb_id, 'target_chains': t_chains, 'partner_chains': p_chains,
        'target_residues': [res_key(r) for c in tgt for r in c if r.id[0] == ' '],
        'interface_contact_5A': sorted(contact),
        'interface_dsasa': dsasa_set,
        'interface_consensus': consensus,
        'partner_interface_5A': sorted(partner_iface),
        'dsasa_per_residue': {k: round(v, 2) for k, v in dsasa.items() if v > 0.1},
        'target_seq': {c.id: seq_of(c) for c in tgt},
        'partner_seq': {c.id: seq_of(c) for c in par},
        'target_gaps': {c.id: numbering_gaps(c) for c in tgt},
        'target_numbering': {c.id: [min(r.id[1] for r in c if r.id[0] == ' '),
                                    max(r.id[1] for r in c if r.id[0] == ' ')] for c in tgt},
        'partner_len': sum(1 for c in par for r in c if r.id[0] == ' '),
        'partner_helix_frac': {c.id: helix_fraction(c) for c in par},
        'flags': flags,
    }
    json.dump(info, open(f'{out_dir}/interface_residues.json', 'w'), indent=2)
    return info

print('Helpers defined.')

Helpers defined.


In [20]:
# ── Cell 5: Propose chain assignments and STOP for review ───────────────────────
# Writes config/chain_assignments.json. If it already exists your edits are kept
# (set OVERWRITE = True to regenerate). Edit chains by hand there, set "status": "CONFIRMED" or "EXCLUDED".
OVERWRITE = False
ASSIGN_PATH = f'{CONFIG}/chain_assignments.json'
assign = json.load(open(ASSIGN_PATH)) if (os.path.exists(ASSIGN_PATH) and not OVERWRITE) else {}

for _, row in pairs.iterrows():
    key, pid = str(row['id']), row['pdb_id'].upper()
    # Skip entries that have already been confirmed, auto-assigned, or manually excluded
    if key in assign and assign[key].get('status') in ('CONFIRMED', 'AUTO', 'EXCLUDED'):
        continue
    entry = {'pdb_id': pid, 'target': row['target_protein'], 'partner': row['partner_to_predict'],
             'status': 'UNRESOLVED', 'target_chains': [], 'partner_chains': [], 'entities': [], 'notes': []}
    try:
        ents = fetch_entities(pid)
        entry['entities'] = [f"{e['desc']} | {e['type']} | {e['organism']} | chains {e['chains']}" for e in ents]
        if row['id'] not in ALIASES:
            entry['notes'].append('no aliases defined')
        else:
            tkw, pkw = ALIASES[row['id']]
            if row['id'] in SWAP_IDS:
                tkw, pkw = pkw, tkw
            model = load_model0(download_structure(pid))
            prop, msg = propose_chains(model, ents, tkw, pkw)
            if prop:
                entry.update(status='AUTO', target_chains=prop['target_chains'], partner_chains=prop['partner_chains'])
                entry['notes'] += prop['notes'] + [f"contacts(4.5A)={prop['contacts_4.5A']}"]
            else:
                entry['notes'].append(msg)
    except Exception as e:
        entry['notes'].append(f'error: {e}')
    assign[key] = entry

json.dump(assign, open(ASSIGN_PATH, 'w'), indent=2)

print('REVIEW THESE (compare target/partner names with the entity list):\n')
for k, v in assign.items():
    print(f"[{k}] {v['pdb_id']}  {v['status']}  target={v['target']} {v['target_chains']}  partner={v['partner']} {v['partner_chains']}")
    for e in v['entities']: print('       entity:', e)
    for n in v['notes']:    print('       note:', n)
    print()
unresolved = [k for k, v in assign.items() if v['status'] == 'UNRESOLVED']
print('UNRESOLVED:', unresolved or 'none')
print(f'\nIf any assignment is wrong: edit {ASSIGN_PATH} (chain lists), set status to CONFIRMED or EXCLUDED, re-run this cell, then run Cell 6.')

REVIEW THESE (compare target/partner names with the entity list):

[4] 1JSU  CONFIRMED  target=CDK2 ['A', 'B']  partner=CDKN1B (p27) ['C']
       entity: P27 | Protein | Homo sapiens | chains ['C']
       entity: CYCLIN A | Protein | Homo sapiens | chains ['B']
       entity: CYCLIN-DEPENDENT KINASE-2 | Protein | Homo sapiens | chains ['A']
       note: contacts(4.5A)=335
       note: manual: target = CDK2 (A) + cyclin A (B)

[5] 1YCR  AUTO  target=MDM2 ['A']  partner=TP53 peptide ['B']
       entity: P53 | Protein | Homo sapiens | chains ['B']
       entity: MDM2 | Protein | Homo sapiens | chains ['A']
       note: contacts(4.5A)=253

[6] 1JM7  AUTO  target=BRCA1 ['A']  partner=BARD1 ['B']
       entity: BRCA1-ASSOCIATED RING DOMAIN PROTEIN 1 | Protein | Homo sapiens | chains ['B']
       entity: BREAST CANCER TYPE 1 SUSCEPTIBILITY PROTEIN | Protein | Homo sapiens | chains ['A']
       note: contacts(4.5A)=398

[7] 1N0W  AUTO  target=RAD51 ['A']  partner=BRCA2 (BRC4) ['B']
       enti

In [21]:
# Cell 5b: corrections
assign = json.load(open(ASSIGN_PATH))
assign['4'].update(target_chains=['A', 'B'], status='CONFIRMED')   # A = CDK2, B = cyclin A
assign['4']['notes'].append('manual: target = CDK2 (A) + cyclin A (B)')
assign['22']['partner'] = 'RGS14 GoLoco (1KJY is not AGS3)'
assign['8']['partner']  = 'p21 PIP peptide (POLD1 proxy)'
assign['45'].update(status='EXCLUDED'); assign['45']['notes'].append('1N7E is apo, no peptide')
json.dump(assign, open(ASSIGN_PATH, 'w'), indent=2)

def n_mse(pdb):   # selenomethionine residues the cleaning step would have dropped
    n = 0
    for line in open(f'{LOCAL_RAW}/{pdb}.cif'):
        f = line.split()
        if line.startswith('HETATM') and len(f) > 5 and f[5] == 'MSE' and f[3] == 'CA': n += 1
    return n
for k, v in assign.items():
    if v['status'] in ('AUTO', 'CONFIRMED'): print(k, v['pdb_id'], 'MSE residues:', n_mse(v['pdb_id']))

4 1JSU MSE residues: 0
5 1YCR MSE residues: 0
6 1JM7 MSE residues: 0
7 1N0W MSE residues: 10
8 1AXC MSE residues: 0
10 2XA0 MSE residues: 0
11 1G5J MSE residues: 0
12 3EZQ MSE residues: 0
13 1NW9 MSE residues: 0
20 4G0N MSE residues: 0
22 1KJY MSE residues: 0
24 1J2J MSE residues: 0
26 1FOS MSE residues: 0
27 1NKP MSE residues: 0
28 1KDX MSE residues: 0
30 4GQ6 MSE residues: 0
33 1LM8 MSE residues: 0
34 2FLU MSE residues: 0
37 1JDH MSE residues: 0
38 1T01 MSE residues: 0


In [22]:
# Cell 5c: keep selenomethionine as methionine when writing cleaned files
class KeepChains(PDB.Select):
    def __init__(self, chains): self.chains = set(chains)
    def accept_chain(self, chain): return chain.id in self.chains
    def accept_residue(self, residue): return residue.id[0] == ' ' or residue.get_resname() == 'MSE'
    def accept_atom(self, atom): return atom.get_altloc() in (' ', 'A')

def _mse_to_met(path):
    out = []
    for line in open(path):
        if line.startswith('HETATM') and line[17:20] == 'MSE':
            line = 'ATOM  ' + line[6:17] + 'MET' + line[20:]
            if line[12:16].strip() == 'SE':
                line = line[:12] + ' SD ' + line[16:76] + ' S' + line[78:]
        out.append(line)
    open(path, 'w').writelines(out)

def write_chains(model_struct, chains, out_path):
    io = PDB.PDBIO(); io.set_structure(model_struct); io.save(out_path, KeepChains(chains))
    _mse_to_met(out_path)

In [23]:
# ── Cell 6: Build the gold standard for every AUTO / CONFIRMED assignment ──────
assign = json.load(open(ASSIGN_PATH))
summary, failed = [], {}
for key, v in assign.items():
    if v['status'] == 'EXCLUDED':
        continue  # Safely skip excluded entries (like apo structures)
    if v['status'] not in ('AUTO', 'CONFIRMED'):
        failed[key] = 'unresolved'; continue
    out_dir = f'{GOLD}/{key}_{v["pdb_id"]}'
    try:
        info = build_gold(v['pdb_id'], v['target_chains'], v['partner_chains'], out_dir)
        orgs = sorted({e.split(' | ')[2] for e in v['entities'] if e.split(' | ')[2] != 'None'})
        summary.append({
            'id': int(key), 'pdb_id': v['pdb_id'], 'target': v['target'], 'partner': v['partner'],
            'target_chains': ','.join(info['target_chains']), 'partner_chains': ','.join(info['partner_chains']),
            'n_target_res': len(info['target_residues']), 'n_iface_contact': len(info['interface_contact_5A']),
            'n_iface_dsasa': len(info['interface_dsasa']), 'n_iface_consensus': len(info['interface_consensus']),
            'partner_len': info['partner_len'],
            'partner_helix_frac': max([x for x in info['partner_helix_frac'].values() if x is not None], default=None),
            'has_gaps': any(info['target_gaps'].values()), 'flags': ';'.join(info['flags']),
            'organisms': '; '.join(orgs)})
        print(f"[{key}] {v['pdb_id']}: consensus interface {len(info['interface_consensus'])} res, "
              f"partner {info['partner_len']} aa {info['flags'] or ''}")
    except Exception as e:
        failed[key] = str(e); print(f'[{key}] {v["pdb_id"]}: FAILED - {e}')

df = pd.DataFrame(summary).sort_values('id')
df.to_csv(f'{GOLD}/gold_summary.csv', index=False)
print('\n', df.drop(columns=['target', 'partner']).to_string(index=False))
print('\nFailed / skipped:', failed or 'none')

[4] 1JSU: consensus interface 78 res, partner 69 aa 
[5] 1YCR: consensus interface 23 res, partner 13 aa 
[6] 1JM7: consensus interface 32 res, partner 97 aa 
[7] 1N0W: consensus interface 33 res, partner 33 aa 
[8] 1AXC: consensus interface 36 res, partner 18 aa 
[10] 2XA0: consensus interface 29 res, partner 26 aa 
[11] 1G5J: consensus interface 30 res, partner 25 aa 
[12] 3EZQ: consensus interface 28 res, partner 99 aa 
[13] 1NW9: consensus interface 27 res, partner 238 aa 
[20] 4G0N: consensus interface 15 res, partner 78 aa 
[22] 1KJY: consensus interface 51 res, partner 35 aa 
[24] 1J2J: consensus interface 12 res, partner 41 aa 
[26] 1FOS: consensus interface 24 res, partner 59 aa 
[27] 1NKP: consensus interface 37 res, partner 83 aa 
[28] 1KDX: consensus interface 17 res, partner 27 aa 
[30] 4GQ6: consensus interface 29 res, partner 12 aa 
[33] 1LM8: consensus interface 20 res, partner 14 aa 
[34] 2FLU: consensus interface 17 res, partner 16 aa 
[37] 1JDH: consensus interface 5

In [24]:
# ── Cell 7: Phase 0 manifest (versions + seed) and next-step checklist ────────
import Bio, scipy
manifest = {
    'created': time.strftime('%Y-%m-%d %H:%M:%S'), 'seed': SEED,
    'python': platform.python_version(), 'biopython': Bio.__version__,
    'numpy': np.__version__, 'scipy': scipy.__version__, 'pandas': pd.__version__,
    'contact_cutoff_A': CONTACT_CUTOFF, 'dsasa_min_A2': DSASA_MIN,
    'swap_ids': sorted(SWAP_IDS), 'n_gold_built': len(summary), 'n_failed': len(failed),
    'failed': failed,
    'note': 'RFdiffusion / ProteinMPNN / Boltz versions to be added when those phases are set up.'
}
json.dump(manifest, open(f'{MANIFEST}/phase0_1_manifest.json', 'w'), indent=2)
print(json.dumps(manifest, indent=2))
print('\nBefore Phase 2, check by hand:')
print('  - every PDB in gold_summary.csv is the complex you intended (names + organisms)')
print('  - rows flagged SMALL_INTERFACE / TINY_PARTNER / NO_CONTACT')
print('  - peptide partners: confirm the peptide chain is actually present in the structure')

{
  "created": "2026-10-10 17:05:42",
  "seed": 42,
  "python": "3.13.16",
  "biopython": "1.88",
  "numpy": "2.1.3",
  "scipy": "1.16.3",
  "pandas": "2.2.3",
  "contact_cutoff_A": 5.0,
  "dsasa_min_A2": 1.0,
  "swap_ids": [
    28,
    45
  ],
  "n_gold_built": 20,
  "n_failed": 0,
  "failed": {},
  "note": "RFdiffusion / ProteinMPNN / Boltz versions to be added when those phases are set up."
}

Before Phase 2, check by hand:
  - every PDB in gold_summary.csv is the complex you intended (names + organisms)
  - rows flagged SMALL_INTERFACE / TINY_PARTNER / NO_CONTACT
  - peptide partners: confirm the peptide chain is actually present in the structure
